# Instagram Engagement Analysis

## Objectives

1. The primary objective of this notebook is to execute the data engineering and feature transformation pipeline for raw social media performance data. Since the underlying dataset lacks chronological/temporal tracking, the analytical scope is directed toward a **Structural Content Optimization Matrix**.

### Key Deliverables:
1. **Diagnostic Profiling & Data Cleaning:** Investigate dataset schema, handle missing or anomalous values, and standardize variable formats.
2. **Feature Engineering:** Synthesize the `total_interactions` metric by aggregating individual engagement layers (Likes, Shares, and Comments) to establish a baseline for organic volume.
3. **Statistical Data Binning:** Apply categorical binning to transform the continuous numeric variable `post_length` (character counts) into discrete, actionable segments (`Short`, `Medium`, `Long`) for matrix cross-tabulation.
4. **Analytical Export:** Output a refined, clean dataset structured specifically to power high-performance reporting and relational modeling within Power BI.

2. Data Dictionary:
   
* 'platform' - Post platform (Instagram, Facebook).

* 'post-type' - Whether it's text, video or image.

* 'post-length' - Number of characters

* 'views' - Amount of views

* 'likes' - Amount of likes

* 'comments' - Amount of comments

* 'shares' - Amount of shares

* 'engagement_rate' - Ratio between interactions ('likes' + 'comments' + 'shares') and 'views'.

## 1. Data Acquisition and Exploratory Diagnosis
Raw dataset ingestion and initial structural audit to identify missing values, data type missmatches and anomalies before moving into pre-procesing.

In [12]:
import pandas as pd
import numpy as np

In [13]:
# Loading Dataset
df = pd.read_csv("../data/instagram_engagement.csv")

In [14]:
# Checking data types and looking for missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   platform         10000 non-null  object 
 1   post_type        10000 non-null  object 
 2   post_length      10000 non-null  int64  
 3   views            10000 non-null  int64  
 4   likes            10000 non-null  int64  
 5   comments         10000 non-null  int64  
 6   shares           10000 non-null  int64  
 7   engagement_rate  10000 non-null  float64
dtypes: float64(1), int64(5), object(2)
memory usage: 625.1+ KB


## 2. Data Pre-processing
During the exploratory diagnosis, the dataset revealed 100% data completness with zero missing values. Since the data set lacks temporal atributes, our analytical strategy will pivot from chronological evolution to Content Matrix Optimization.

### 2.1. Empirical Justification for Bin Edges


Before applying thresholds, we evaluate the statistical distribution (descriptive statistics) of the continuous variable `post_length` to ensure our bin boundaries reflect the actual density of the dataset.

Following the statistical distribution analysis, we implement:
1. **Data Binning on `post_length`**: Transforming the continuous variable into a discrete attribute (`length_category`).
2. **Feature Aggregation (`total_interactions`)**: Combining `likes`, `comments`, and `shares` into a single performance metric.

In [15]:
# Statistical summary of the post length to define binning edges
df['post_length'].describe()

count    10000.000000
mean        62.359800
std         33.264888
min          5.000000
25%         34.000000
50%         62.000000
75%         91.000000
max        119.000000
Name: post_length, dtype: float64

### 2.2. Threshold Selection Based on Descriptive Statistics

The statistical summary indicates that 'post_length' ranges from 5 to 119 characters.

The binning boundaries are defined based on the quartiles:
* **Short**: < 40 characters
* **Medium**: 40 to 80 characters
* **Long**: ≥ 80 characters

In [16]:
# 1. Applying Data Binning
bin_edges = [0,40,80, float('inf')]
bin_labels = ['Short', 'Medium', 'Long']
df['length category'] = pd.cut(df['post_length'], bins = bin_edges, labels = bin_labels, right = False)

# 2. Feature Aggregation
df['total_interactions'] = df['likes'] + df['comments'] + df['shares']

# 3. Verify the count distribution of our new bins
df['length category'].value_counts()

length category
Long      3517
Medium    3492
Short     2991
Name: count, dtype: int64

## 3. Data Export

With the feature engineering and data binning complete, we perform a final structural integrity check on the modified dataframe.

This establishes a clean hand-off, ensuring that our downstream Power BI dashboard ingests a highly optimized data structure, reducing visual rendering latency and eliminating the need for complex transformations on Power Query.

In [17]:
# 1. Inspect the first 5 rows to ensure new columns look correct
print('DataFrame Preview')
df.head()

DataFrame Preview


,platform,post_type,post_length,views,likes,comments,shares,engagement_rate,length category,total_interactions
0,Facebook,Text,62,91660,2968,276,346,0.039166,Medium,3590
1,Instagram,Video,104,113115,4164,632,406,0.045989,Long,5202
2,Facebook,Video,46,36043,3125,188,100,0.094692,Medium,3413
3,Facebook,Image,39,124886,5970,948,578,0.060023,Short,7496
4,Instagram,Video,42,82831,8212,1104,334,0.116502,Medium,9650


In [18]:
# 2. Inspect the first 5 rows to ensure new columns look correct
print('DataFrame Preview')
df.tail()

DataFrame Preview


,platform,post_type,post_length,views,likes,comments,shares,engagement_rate,length category,total_interactions
9995,Facebook,Image,65,42330,1436,158,122,0.040539,Medium,1716
9996,Instagram,Text,119,254309,12274,2226,1089,0.061299,Long,15589
9997,Twitter,Image,17,48415,4126,503,581,0.107611,Short,5210
9998,Twitter,Video,16,63006,5504,529,499,0.103673,Short,6532
9999,Facebook,Image,33,183199,17318,1292,971,0.106884,Short,19581


In [19]:
# 3. Final verification of data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   platform            10000 non-null  object  
 1   post_type           10000 non-null  object  
 2   post_length         10000 non-null  int64   
 3   views               10000 non-null  int64   
 4   likes               10000 non-null  int64   
 5   comments            10000 non-null  int64   
 6   shares              10000 non-null  int64   
 7   engagement_rate     10000 non-null  float64 
 8   length category     10000 non-null  category
 9   total_interactions  10000 non-null  int64   
dtypes: category(1), float64(1), int64(6), object(2)
memory usage: 713.1+ KB


In [21]:
# 4. Export to a clean CSV file
df.to_csv('../data/instagram_processed_data.csv', index = False)
print('The clean file was exported into the "data" folder')

The clean file was exported into the "data" folder
